# Lesson 1.3 — Applications, Tiers, and Nodes

## Learning Objectives
- Explain the three-level topology: Application → Tier → Node
- Understand how agents map to tiers and nodes
- Navigate the Application topology in the Controller UI
- Use the REST API to list tiers and nodes for an application

**Reference**: [Tiers and Nodes](https://help.splunk.com/en/appdynamics-saas/application-performance-monitoring/26.6.0/tiers-and-nodes) | [Business Applications](https://help.splunk.com/en/appdynamics-saas/application-performance-monitoring/26.6.0/business-applications)

## 1. The Three-Level Topology

AppDynamics organizes monitored infrastructure in a hierarchy:

```
APPLICATION
├── Tier A  (e.g., "WebFrontend")
│   ├── Node A1  (web-server-01)
│   ├── Node A2  (web-server-02)
│   └── Node A3  (web-server-03)
├── Tier B  (e.g., "OrderService")
│   ├── Node B1  (order-svc-pod-1)
│   └── Node B2  (order-svc-pod-2)
└── Tier C  (e.g., "DB-Backend")   ← auto-detected, no agent needed
    └── (virtual backend node)
```

| Level | Definition | Example |
|-------|------------|----------|
| **Application** | A logical business system — all the components that together deliver a feature | "StoreFront" |
| **Tier** | A group of nodes performing the **same function** | "CheckoutService", "InventoryAPI" |
| **Node** | A single running process or container instance | `checkout-pod-abc123`, `web-srv-01` |

In [ ]:
# Let's model a real-world application topology in Python

from dataclasses import dataclass, field
from typing import List

@dataclass
class Node:
    name: str
    host: str
    agent_type: str  # JAVA, DOTNET, NODEJS, PYTHON, etc.
    is_up: bool = True

@dataclass
class Tier:
    name: str
    nodes: List[Node] = field(default_factory=list)

@dataclass
class Application:
    name: str
    tiers: List[Tier] = field(default_factory=list)

    def print_topology(self):
        print(f"Application: {self.name}")
        for tier in self.tiers:
            print(f"  Tier: {tier.name} ({len(tier.nodes)} node(s))")
            for node in tier.nodes:
                status = "UP" if node.is_up else "DOWN"
                print(f"    └─ Node: {node.name}  Host: {node.host}  Agent: {node.agent_type}  [{status}]")

# Build a sample StoreFront application topology
storefront = Application(
    name="StoreFront",
    tiers=[
        Tier("WebFrontend", [
            Node("web-01", "10.0.1.1", "NODEJS"),
            Node("web-02", "10.0.1.2", "NODEJS"),
        ]),
        Tier("OrderService", [
            Node("order-svc-1", "10.0.2.1", "JAVA"),
            Node("order-svc-2", "10.0.2.2", "JAVA"),
        ]),
        Tier("InventoryAPI", [
            Node("inventory-1", "10.0.3.1", "PYTHON"),
        ]),
        Tier("OrderDB", [
            Node("mysql-primary", "10.0.4.1", "DB"),
        ]),
    ]
)

storefront.print_topology()

## 2. How Agents Create This Topology

When you install an **App Server Agent** on a process, you tell the agent:
- Which **Application** it belongs to (via `app-name` configuration)
- Which **Tier** it belongs to (via `tier-name` configuration)

The agent then **registers itself** with the Controller, creating the Node entry automatically.

**Agent configuration example** (Java agent — `controller-info.xml`):
```xml
<controller-info>
  <controller-host>acme-corp.saas.appdynamics.com</controller-host>
  <controller-port>443</controller-port>
  <account-name>acme-corp</account-name>
  <account-access-key>your-access-key-here</account-access-key>
  <application-name>StoreFront</application-name>
  <tier-name>OrderService</tier-name>
  <node-name>order-svc-1</node-name>
  <ssl-enabled>true</ssl-enabled>
</controller-info>
```

> **Best Practice**: Use consistent naming conventions. Bad tier names like "app1-instance" make dashboards unreadable. Good names: "CheckoutService", "ProductCatalogAPI", "AuthService".

## 3. Backend Detection — Automatic Tier Discovery

AppDynamics automatically detects **remote services** (backends) that your application calls — even without installing an agent there:

- **Databases**: MySQL, Oracle, PostgreSQL, MSSQL, MongoDB, etc.
- **Message queues**: Kafka, RabbitMQ, JMS
- **External HTTP services**: REST APIs, microservices
- **Cache systems**: Redis, Memcached

These appear in the topology as **Backend** nodes — shown with a different icon in the Flow Map. Backend nodes track call count, response time, and errors **from the caller's perspective**.

In [ ]:
# REST API: List tiers for an application
# Requires Lesson 1.2 credentials to be configured

import os, requests, json

CONTROLLER_URL = os.getenv("APPD_CONTROLLER_URL", "https://your-account.saas.appdynamics.com")
ACCOUNT_NAME   = os.getenv("APPD_ACCOUNT",        "your-account")
USERNAME       = os.getenv("APPD_USER",            "admin")
PASSWORD       = os.getenv("APPD_PASSWORD",        "")

def appd_get(endpoint):
    url = f"{CONTROLLER_URL}/controller/rest/{endpoint}"
    resp = requests.get(url, auth=(f"{USERNAME}@{ACCOUNT_NAME}", PASSWORD),
                        params={"output": "JSON"}, timeout=30)
    resp.raise_for_status()
    return resp.json()

# TASK: Replace 0 with your application ID (from Lesson 1.2)
APP_ID = int(os.getenv("APPD_APP_ID", "0")) or None  # TODO: set your app ID

if APP_ID:
    try:
        tiers = appd_get(f"applications/{APP_ID}/tiers")
        print(f"Tiers in App ID {APP_ID}:")
        for t in tiers:
            print(f"  ID={t['id']} | Tier={t['name']} | AgentType={t.get('agentType','?')} | NodeCount={t.get('numberOfNodes','?')}")
    except Exception as e:
        print(f"API Error: {e}")
else:
    print("APP_ID not set. Set the APPD_APP_ID environment variable or replace APP_ID above.")
    print()
    print("Sample expected output:")
    sample_tiers = [
        {"id": 101, "name": "WebFrontend",  "agentType": "NODEJS", "numberOfNodes": 2},
        {"id": 102, "name": "OrderService", "agentType": "JAVA",   "numberOfNodes": 2},
        {"id": 103, "name": "InventoryAPI", "agentType": "PYTHON", "numberOfNodes": 1},
    ]
    for t in sample_tiers:
        print(f"  ID={t['id']} | Tier={t['name']} | AgentType={t['agentType']} | NodeCount={t['numberOfNodes']}")

In [ ]:
# REST API: List nodes within a tier

# TASK: Replace 0 with a real tier ID from the output above
TIER_ID = 0  # TODO: set a real tier ID

if APP_ID and TIER_ID:
    try:
        nodes = appd_get(f"applications/{APP_ID}/tiers/{TIER_ID}/nodes")
        print(f"Nodes in Tier ID {TIER_ID}:")
        for n in nodes:
            print(f"  Node: {n['name']}  Host: {n.get('machineName','?')}  Agent Type: {n.get('agentType','?')}")
    except Exception as e:
        print(f"API Error: {e}")
else:
    print("Set APP_ID and TIER_ID to list nodes.")
    print("Sample output:")
    print("  Node: order-svc-1  Host: ip-10-0-2-1  Agent Type: JAVA")
    print("  Node: order-svc-2  Host: ip-10-0-2-2  Agent Type: JAVA")

## 4. Navigating the Flow Map in the UI

The **Flow Map** is the most important view in the AppDynamics UI — it shows your entire application topology with real-time health indicators.

### Steps to view your Flow Map:
1. Log in to the Controller UI
2. Click **Applications** in the left navigation
3. Click your application name (e.g., "StoreFront")
4. The **Application Dashboard** opens with a Flow Map showing:
   - Tier boxes (each tier is a box)
   - Arrows showing call relationships between tiers
   - Color coding: **Green** = healthy, **Yellow** = warning, **Red** = critical
   - Numbers on arrows: calls per minute (throughput)

### What to look for:
- **Red tiers** → something is violating a health rule
- **Thick arrows** → high call volume between services
- **Database backends** appear with a cylinder icon
- **Remote HTTP services** appear with a globe icon

In [ ]:
# EXERCISE 1.3: Build a topology dictionary from API data
#
# TASK: Write a function that takes an app_id and returns a dict:
#   { tier_name: [node_name, ...], ... }
# Use the appd_get() helper defined above.

def get_app_topology(app_id: int) -> dict:
    """
    Returns a dict mapping tier names to lists of node names.
    Example: {'WebFrontend': ['web-01', 'web-02'], 'OrderService': ['order-1']}
    """
    topology = {}
    # TODO: Implement using appd_get()
    # Step 1: Get all tiers for app_id
    # Step 2: For each tier, get its nodes
    # Step 3: Build and return the dict
    return topology

# Test with your app (or use the sample below)
if APP_ID:
    topology = get_app_topology(APP_ID)
    for tier, nodes in topology.items():
        print(f"  {tier}: {nodes}")
else:
    # Sample reference output
    sample = {
        "WebFrontend":  ["web-01", "web-02"],
        "OrderService": ["order-svc-1", "order-svc-2"],
        "InventoryAPI": ["inventory-1"]
    }
    for tier, nodes in sample.items():
        print(f"  {tier}: {nodes}")

## Summary

- AppDynamics uses a **three-level hierarchy**: Application → Tier → Node
- **Applications** are logical business systems; **Tiers** group nodes by function; **Nodes** are individual processes
- Agents are configured with `app-name`, `tier-name`, and optionally `node-name`
- **Backends** (databases, queues, external services) are auto-detected without needing an agent
- The **Flow Map** visualizes this topology with real-time health indicators
- The REST API endpoints `applications/{id}/tiers` and `applications/{id}/tiers/{id}/nodes` expose the topology programmatically

---

**Next**: [Lesson 1.4 — Business Transactions](lesson-1.4-business-transactions.ipynb)